# Full Experiment: Uncertainty Quantification on FB15k-237

**GPU:** A100 recommended (Colab Pro)
**Test:** Full test set (20,466 triples)

**Models:**
1. DistMult (baseline)
2. DistMult + MC Dropout
3. GGPN (Graph Gaussian Process Network)
4. GP-KGE (Our method)

**Metrics:**
- Link Prediction: MRR, Hits@1, Hits@10
- Calibration: ECE, Brier Score
- OOD Detection: AUROC

---
## Setup
---

In [ ]:
# Clone and install (force re-clone to get latest code)
!rm -rf /content/kg-bayesian-prior
!git clone https://github.com/ChorokLeeDev/kg-bayesian-prior.git /content/kg-bayesian-prior
!pip install -q torch-geometric gpytorch networkx pandas tqdm scikit-learn

import os, sys
os.chdir('/content/kg-bayesian-prior')
sys.path.insert(0, '/content/kg-bayesian-prior')

In [ ]:
import gc, json, warnings
import torch
import torch.nn.functional as F
import numpy as np
from tqdm.notebook import tqdm

from src.data import load_fb15k237
from src.models import DistMult, GPKGE
from src.models.ggpn import GGPN
from src.models.uncertain_kge import MCDropoutKGE
from src.utils.training import set_seed, NegativeSampler
from src.evaluation.calibration import expected_calibration_error, brier_score
from src.evaluation.ood_detection import compute_auroc, create_ood_dataset

warnings.filterwarnings('ignore')
set_seed(42)
device = "cuda"
print(f"GPU: {torch.cuda.get_device_name(0)}")

train_data, _, test_data = load_fb15k237()
print(f"Data: {len(train_data):,} train, {len(test_data):,} test")

neg_sampler = NegativeSampler(train_data.num_entities, num_negatives=10)
neg_sampler.set_true_triples(train_data.triples)

results = {}

In [ ]:
# Helper functions
def clear_mem():
    gc.collect()
    torch.cuda.empty_cache()

def full_evaluate(model, name, use_forward=False):
    """FULL test set evaluation - no sampling"""
    print(f"\nEvaluating {name} on FULL test ({len(test_data):,} triples)...")
    model.eval()
    
    # === MRR - FULL ===
    ranks = []
    with torch.no_grad():
        for i in tqdm(range(0, len(test_data), 200), desc="MRR", leave=False):
            batch = test_data.triples[i:i+200]
            h, r, t = [torch.tensor(batch[:,j], device=device) for j in range(3)]
            if hasattr(model, 'score_tails'):
                scores = model.score_tails(h, r)
            elif hasattr(model, 'base_model') and hasattr(model.base_model, 'score_tails'):
                scores = model.base_model.score_tails(h, r)
            else:
                all_e = torch.arange(train_data.num_entities, device=device)
                scores = torch.stack([model(h[j].expand(train_data.num_entities),
                                           r[j].expand(train_data.num_entities), all_e) for j in range(len(h))])
            target = scores[torch.arange(len(t), device=device), t]
            ranks.extend(((scores > target.unsqueeze(1)).sum(1) + 1).cpu().tolist())
    
    ranks = torch.tensor(ranks, dtype=torch.float)
    mrr = (1/ranks).mean().item()
    h1 = (ranks <= 1).float().mean().item()
    h10 = (ranks <= 10).float().mean().item()
    
    # === ECE - FULL ===
    pos = test_data.triples
    neg = np.array([[h, r, np.random.randint(train_data.num_entities)] for h,r,t in pos])
    all_t = np.vstack([pos, neg])
    labels = np.concatenate([np.ones(len(pos)), np.zeros(len(neg))])
    
    confs = []
    with torch.no_grad():
        for i in tqdm(range(0, len(all_t), 1024), desc="ECE", leave=False):
            batch = all_t[i:i+1024]
            h, r, t = [torch.tensor(batch[:,j], device=device) for j in range(3)]
            if use_forward:
                scores = model(h, r, t)
            elif hasattr(model, 'base_model'):
                scores = model.base_model.score_triple(h, r, t)
            elif hasattr(model, 'score_triple'):
                scores = model.score_triple(h, r, t)
            else:
                scores = model(h, r, t)
            confs.append(torch.sigmoid(scores).cpu().numpy())
    conf = np.concatenate(confs)
    ece, _ = expected_calibration_error(conf, labels)
    brier = brier_score(conf, labels)
    
    # === AUROC - FULL ===
    ood_t = create_ood_dataset(train_data, test_data, "random", len(test_data))
    
    def get_unc(triples):
        uncs = []
        with torch.no_grad():
            for i in range(0, len(triples), 1024):
                batch = triples[i:i+1024]
                h, r, t = [torch.tensor(batch[:,j], device=device) for j in range(3)]
                if hasattr(model, 'predict_with_uncertainty'):
                    pred = model.predict_with_uncertainty(h, r, t)
                    unc = pred.get('total', pred.get('epistemic', torch.zeros(len(h)))) if isinstance(pred, dict) else pred[1]
                elif hasattr(model, 'predict_with_mc_samples'):
                    _, var = model.predict_with_mc_samples(h, r, t, num_samples=10)
                    unc = var
                else:
                    if use_forward:
                        s = model(h, r, t)
                    elif hasattr(model, 'base_model'):
                        s = model.base_model.score_triple(h, r, t)
                    elif hasattr(model, 'score_triple'):
                        s = model.score_triple(h, r, t)
                    else:
                        s = model(h, r, t)
                    p = torch.sigmoid(s)
                    unc = -p * torch.log(p + 1e-10) - (1-p) * torch.log(1-p + 1e-10)
                uncs.append(unc.cpu().numpy())
        return np.concatenate(uncs)
    
    auroc = compute_auroc(get_unc(test_data.triples), get_unc(ood_t))
    
    results[name] = {"mrr": mrr, "hits@1": h1, "hits@10": h10, 
                     "ece": ece, "brier": brier, "auroc": auroc}
    print(f"{name}: MRR={mrr:.4f}, H@1={h1:.4f}, H@10={h10:.4f}, ECE={ece:.4f}, AUROC={auroc:.4f}")
    return results[name]

print("Ready!")

---
## Model 1: DistMult
---

In [ ]:
print("="*50 + "\nMODEL 1/4: DistMult (BCE Loss)\n" + "="*50)
clear_mem()

model = DistMult(train_data.num_entities, train_data.num_relations, embedding_dim=200).to(device)
opt = torch.optim.Adam(model.parameters(), lr=0.001)

for ep in (pbar := tqdm(range(50), desc="DistMult")):
    model.train(); loss_sum, n = 0, 0
    for st in range(0, len(train_data), 1024):
        pos = torch.tensor(train_data.triples[st:st+1024], device=device)
        neg = pos.clone()
        neg[:,2] = torch.randint(0, train_data.num_entities, (len(pos),), device=device)
        opt.zero_grad()
        pos_s = model(pos[:,0], pos[:,1], pos[:,2])
        neg_s = model(neg[:,0], neg[:,1], neg[:,2])
        # BCE Loss (same as GP-KGE for fair comparison)
        loss = F.binary_cross_entropy_with_logits(
            torch.cat([pos_s, neg_s]), 
            torch.cat([torch.ones_like(pos_s), torch.zeros_like(neg_s)]))
        loss.backward(); opt.step()
        loss_sum += loss.item(); n += 1
    pbar.set_postfix(loss=f"{loss_sum/n:.4f}")

full_evaluate(model, "DistMult")
del model; clear_mem()

---
## Model 2: MCDropout
---

In [ ]:
print("="*50 + "\nMODEL 2/4: MCDropout (BCE Loss)\n" + "="*50)
clear_mem()

base = DistMult(train_data.num_entities, train_data.num_relations, embedding_dim=200, dropout=0.3)
model = MCDropoutKGE(base, num_samples=20).to(device)
opt = torch.optim.Adam(model.parameters(), lr=0.001)

for ep in (pbar := tqdm(range(50), desc="MCDropout")):
    model.train(); loss_sum, n = 0, 0
    for st in range(0, len(train_data), 1024):
        pos = torch.tensor(train_data.triples[st:st+1024], device=device)
        neg = pos.clone()
        neg[:,2] = torch.randint(0, train_data.num_entities, (len(pos),), device=device)
        opt.zero_grad()
        pos_s = model.base_model(pos[:,0], pos[:,1], pos[:,2])
        neg_s = model.base_model(neg[:,0], neg[:,1], neg[:,2])
        # BCE Loss (same as GP-KGE for fair comparison)
        loss = F.binary_cross_entropy_with_logits(
            torch.cat([pos_s, neg_s]), 
            torch.cat([torch.ones_like(pos_s), torch.zeros_like(neg_s)]))
        loss.backward(); opt.step()
        loss_sum += loss.item(); n += 1
    pbar.set_postfix(loss=f"{loss_sum/n:.4f}")

full_evaluate(model, "MCDropout")
del model, base; clear_mem()

---
## Model 3: GGPN
---

In [ ]:
print("="*50 + "\nMODEL 3/4: GGPN (Minimal params)\n" + "="*50)
clear_mem()

model = GGPN(train_data.num_entities, train_data.num_relations*2,
             embedding_dim=50, hidden_dim=50, num_layers=1, num_rff=20).to(device)
model.set_graph(train_data)
opt = torch.optim.Adam(model.parameters(), lr=0.001)

for ep in (pbar := tqdm(range(50), desc="GGPN")):
    model.train(); loss_sum, n = 0, 0
    for st in range(0, len(train_data), 512):
        pos = torch.tensor(train_data.triples[st:st+512], device=device)
        neg = neg_sampler(pos).to(device)
        opt.zero_grad()
        loss = model.loss(pos, neg)
        loss = loss['total'] if isinstance(loss, dict) else loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        loss_sum += loss.item(); n += 1
    pbar.set_postfix(loss=f"{loss_sum/n:.4f}")

full_evaluate(model, "GGPN", use_forward=True)
del model; clear_mem()

---
## Model 4: GP-KGE (Ours)
---

In [ ]:
print("="*50 + "\nMODEL 4/4: GP-KGE (Ours)\n" + "="*50)
clear_mem()

from scipy import sparse
from scipy.sparse.linalg import eigsh
from src.kernels.matern_graph import GraphLaplacian

model = GPKGE(
    train_data.num_entities,
    train_data.num_relations,
    embedding_dim=200,
    kernel_type="relation_aware",
    num_inducing=500
).to(device)

# Safe eigendecomposition
print("Safe eigendecomposition...")
kernel = model.kernel
kernel.num_entities = train_data.num_entities
kernel.relation_laplacians = {}

success, failed = 0, 0
for rel_id, adj in tqdm(train_data.relation_adjacencies.items(), desc="Eigendecomp"):
    if adj.nnz < 10:
        continue
    try:
        degrees = np.array(adj.sum(axis=1)).flatten()
        D_inv_sqrt = sparse.diags(1.0 / np.sqrt(np.maximum(degrees, 1e-10)))
        L = sparse.diags(degrees) - adj
        L_norm = D_inv_sqrt @ L @ D_inv_sqrt
        L_norm = (L_norm + L_norm.T) / 2
        k = min(100, L_norm.shape[0] - 2)
        if k < 2:
            continue
        eigvals, eigvecs = eigsh(L_norm, k=k, which='SM', maxiter=1000, tol=1e-4)
        kernel.relation_laplacians[rel_id] = GraphLaplacian(adj.shape[0])
        kernel.relation_laplacians[rel_id].eigenvalues = torch.tensor(eigvals, dtype=torch.float32)
        kernel.relation_laplacians[rel_id].eigenvectors = torch.tensor(eigvecs, dtype=torch.float32)
        success += 1
    except:
        failed += 1
print(f"Eigendecomp: {success} success, {failed} failed")

In [ ]:
# Training
print("\nTraining GP-KGE...")
opt = torch.optim.Adam(model.parameters(), lr=0.001)

for ep in (pbar := tqdm(range(50), desc="GP-KGE")):
    model.train(); loss_sum, n = 0, 0
    for st in range(0, len(train_data), 1024):
        pos = torch.tensor(train_data.triples[st:st+1024], device=device)
        neg = pos.clone()
        neg[:,2] = torch.randint(0, train_data.num_entities, (len(pos),), device=device)
        opt.zero_grad()
        ps = model.score_triple(pos[:,0], pos[:,1], pos[:,2], use_mean=True)
        ns = model.score_triple(neg[:,0], neg[:,1], neg[:,2], use_mean=True)
        loss = F.binary_cross_entropy_with_logits(
            torch.cat([ps, ns]), torch.cat([torch.ones_like(ps), torch.zeros_like(ns)]))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        loss_sum += loss.item(); n += 1
    pbar.set_postfix(loss=f"{loss_sum/n:.4f}")

full_evaluate(model, "GP-KGE")
del model; clear_mem()

---
## Final Results
---

In [ ]:
print("\n" + "="*70)
print("FINAL RESULTS (Full Test Set)")
print("="*70)
print(f"{'Model':<20} {'MRR':>8} {'H@1':>8} {'H@10':>8} {'ECE':>8} {'Brier':>8} {'AUROC':>8}")
print("-"*70)

for name, r in results.items():
    print(f"{name:<20} {r['mrr']:>8.4f} {r['hits@1']:>8.4f} {r['hits@10']:>8.4f} {r['ece']:>8.4f} {r['brier']:>8.4f} {r['auroc']:>8.4f}")

if "GGPN" in results and "GP-KGE" in results:
    ggpn_ece = results["GGPN"]["ece"]
    gpkge_ece = results["GP-KGE"]["ece"]
    improvement = (ggpn_ece - gpkge_ece) / ggpn_ece * 100
    print("\n" + "="*70)
    print("KEY FINDING")
    print("="*70)
    print(f"  GGPN ECE:    {ggpn_ece:.4f}")
    print(f"  GP-KGE ECE:  {gpkge_ece:.4f}")
    print(f"  Improvement: {improvement:.1f}%")

In [ ]:
# Save results
with open('/content/final_results.json', 'w') as f:
    json.dump(results, f, indent=2)
print("Saved to /content/final_results.json")

try:
    from google.colab import files
    files.download('/content/final_results.json')
except:
    pass